In [1]:
import collections
from tqdm import tqdm
from collections import defaultdict
import random
import sys
import os
import numpy as np
import ngrams as ngrams
from entropy import filter_candidates, calculate_entropy
from rollback import rollback_iterative
current_dir = os.getcwd()
os.chdir('bert-base-ver2')
from bert_testing import load_model_and_tokenizer, predict_masked_character
os.chdir(current_dir)

In [2]:
class HangmanLocal(object):
    def __init__(self,  
                use_ngrams=True,
                use_entropy=False,
                use_bert=False,
                use_rollback=False,
                use_noncumulative_frequency=True,
                max_n_of_ngrams=6, 
                max_n_unknowns_for_bert=2,
                ngram_weights=None,
                entropy_params=None,
                rollback_similarity_threshold=0.95,
                ):
        """
        Initialize the HangmanLocal object.
        """
        self.use_entropy = use_entropy
        self.use_ngrams = use_ngrams
        self.use_bert = use_bert
        self.use_rollback = use_rollback
        self.use_noncumulative_frequency = use_noncumulative_frequency
        self.max_n_of_ngrams = max_n_of_ngrams
        self.max_n_unknowns_for_bert = max_n_unknowns_for_bert
        self.rollback_similarity_threshold = rollback_similarity_threshold
        
        self.guessed_letters = []
        self.incorrect_guesses = []

        if ngram_weights is None:
            self.ngram_weights = [1] * max_n_of_ngrams
        else:
            self.ngram_weights = ngram_weights

        if entropy_params is None:
            self.entropy_params = [1, 2, 2]
        else:
            self.entropy_params = entropy_params
        
        # load the full dictionary
        full_dictionary_location = "datasets/words_250000_train_cleaned.txt"
        self.full_dictionary = self.build_dictionary(full_dictionary_location)
        self.full_dictionary_common_letter_sorted = collections.Counter("".join(self.full_dictionary)).most_common()
        self.training_words_set = set(self.full_dictionary) # build training words set to be used for rollback strateg

        # build N-grams
        self.n_grams = ngrams.build_n_gram_from_file(
            full_dictionary_location, 
            max_n_of_ngrams=max_n_of_ngrams, 
            use_noncumulative_frequency=use_noncumulative_frequency
        )
        self.current_dictionary = []

    def guess(self, word):
        # clean the word, replacing "_" with "." to match missing letters
        clean_word = word[::2].replace("_", ".")

        # compute counts
        num_known_letters = len([c for c in clean_word if c != '.'])
        num_unknown_slots = clean_word.count('.')
        num_incorrect_guesses = len(self.incorrect_guesses)

        # extract entropy parameters
        param_num_known_letters, param_num_unknown_slots, param_num_incorrect_guesses = self.entropy_params

        # check whether to use the information entropy method in the current guess round
        use_entropy_for_this_guess = (
            self.use_entropy and
            num_known_letters < param_num_known_letters and
            num_unknown_slots > param_num_unknown_slots and
            num_incorrect_guesses < param_num_incorrect_guesses
            )

        # check whether to use the weighted n-grams method in the current guess round
        use_ngrams_for_this_guess = (
            self.use_ngrams
        )

        # check whether to use the finetuned BERT method in the current guess round
        use_bert_for_this_guess = (
            self.use_bert and
            num_unknown_slots <= self.max_n_unknowns_for_bert
        )

        # Initialize candidate letters list
        candidate_letters = []
        
        # determine which method to use for this guess
        if use_entropy_for_this_guess:
            # Use the information entropy method
            candidates = filter_candidates(self.full_dictionary, clean_word)
            if candidates:
                entropy = calculate_entropy(candidates, self.guessed_letters)
                # calculate_entropy has already excluded the guessed letters
                candidate_letters = sorted(entropy, key=entropy.get, reverse=True)
            else:
                # if no candidate letters, switch to n-grams method (even if not activated)
                # pad the word with '#' and '$' for n-gram calculation
                clean_word_padded = '#' + clean_word + '$'
                # use n-grams to get probabilities of all letters
                ngram_probs = ngrams.get_n_gram_prob(
                    n_grams=self.n_grams,
                    word=clean_word_padded,
                    guessed_letters=self.guessed_letters,
                    ngram_weights=self.ngram_weights
                )
                # sorted by probability, excluding already guessed letters
                sorted_indices = np.argsort(-ngram_probs)
                candidate_letters = []
                for i in sorted_indices:
                    letter = chr(i + 97)  # convert index to letter
                    if letter not in self.guessed_letters:
                        candidate_letters.append(letter)

        elif use_bert_for_this_guess:
            # load bert model and tokenizer
            model, tokenizer, device = load_model_and_tokenizer(use_bert_large=False)

            # prepare already guessed letters for post-processing
            guessed_correct = {c for c in clean_word if c != '.'}
            guessed_incorrect = set(self.incorrect_guesses)

            # mask indices for the unknown letters
            mask_indices = [i for i, c in enumerate(clean_word) if c == '.']

            # predict probabilities for all letters using BERT
            # excluding guessed and correct letters and guessed but incorrect letters
            # extract probabilities from predict_masked_character
            bert_probs = predict_masked_character(
                clean_word, mask_indices, tokenizer, model, device, guessed_correct, guessed_incorrect, verbose=False
            )[1]  # the second return value is the lsit of probabilities
            
            # sort letters based on probabilities
            candidate_letters = sorted(bert_probs, key=bert_probs.get, reverse=True)

        elif use_ngrams_for_this_guess:
            # Use weighted n-grams method
            # Pad word with # at start and $ at end
            clean_word_padded = '#' + clean_word + '$'

            # get n-gram probabilities using the specified ngram_weights
            ngram_probs = ngrams.get_n_gram_prob(
                n_grams=self.n_grams, 
                word=clean_word_padded, 
                guessed_letters=self.guessed_letters, 
                ngram_weights=self.ngram_weights
                )

            # get list of letters sorted by probability, excluding already guessed letters
            sorted_indices = np.argsort(-ngram_probs) # sort and return the sorted indices
            candidate_letters = []
            for i in sorted_indices:
                letter = chr(i + 97)  # convert index to letter
                if letter not in self.guessed_letters:  
                    candidate_letters.append(letter) 

        # fallback to ensure candidate_letters is not empty
        if not candidate_letters:
            # fallback to all unguessed letters if candidate_letters is empty
            candidate_letters = [letter for letter in "abcdefghijklmnopqrstuvwxyz" if letter not in self.guessed_letters]
            if not candidate_letters:
                raise ValueError("No valid candidates available to guess.")

        # check if rollback strategy is enabled
        if self.use_rollback:
            # use rollback strategy and return the next best candidate
            next_letter = rollback_iterative(
                clean_word,
                candidate_letters,
                num_unknown_slots,
                self.guessed_letters,
                self.training_words_set,
                self.rollback_similarity_threshold,
                max_iterations=3,
                enable_two_unknowns=True
            )
            return next_letter
        else:
            return candidate_letters[0]

    ##########################################################
    # You'll likely not need to modify any of the code below #
    ##########################################################
    
    def build_dictionary(self, dictionary_file_location):
        text_file = open(dictionary_file_location, "r")
        full_dictionary = text_file.read().splitlines()
        text_file.close()
        return full_dictionary
                    
    def start_game(self, word_to_guess, practice=True, verbose=True):
        self.guessed_letters = []
        self.incorrect_guesses = []
        self.current_dictionary = self.full_dictionary

        game_id = 1
        word = "_ " * len(word_to_guess)
        letter_remains = len(word_to_guess)
        tries_remains = 6  # Set a fixed number of tries
        if verbose:
            print("Successfully start a new game! Game ID: {0}. # of tries remaining: {1}. Word: {2}.".format(game_id, tries_remains, word))
        while tries_remains > 0:
            # Get guessed letter from user code
            guess_letter = self.guess(word)

            # Append guessed letter to guessed letters field in hangman object
            self.guessed_letters.append(guess_letter)
            if verbose:
                print("Guessing letter: {0}".format(guess_letter))

            if guess_letter in word_to_guess:
                # Replace the underline with the guessed letter
                for i in range(len(word_to_guess)):
                    if word_to_guess[i] == guess_letter:
                        letter_remains -= 1
                        word = word[:2 * i] + guess_letter + word[2 * i + 1:]
                if verbose:
                    print("Successfully guessed letter: {0}. Word: {1}".format(guess_letter, word))
                if letter_remains == 0:
                    if verbose:
                        print("Successfully finished game: {0}".format(game_id))
                    return True
            else:
                self.incorrect_guesses.append(guess_letter)
                tries_remains -= 1  # Decrease the number of tries if the guess was wrong
                if verbose:
                    print("Failed. # of tries remaining: {1}".format(guess_letter, tries_remains))

            if tries_remains == 0:
                if verbose:
                    print("Failed game: {0}. Because of: # of tries exceeded!".format(game_id))
                return False

        return False

In [3]:
# configure the game-dynamics-aware hangman solver
# 3 conditions to satisfy to use information entropy: 
# (1) num_known_letters < 1st param 
# (2) num_unknown_slots > 2nd param 
# (3) num_incorrect_guesses < 3rd param
use_ngrams = False
use_entropy = False
use_bert = True
use_rollback = True
use_noncumulative_frequency = True
max_n_of_ngrams = 6 # current version is fixed to 6, bug with other numbers
max_n_unknowns_for_bert = 100
ngram_weights = [1, 2, 4, 8, 16, 32] # 6 numbers for 6-grams, 5 numbers for 5-grams
entropy_params = [1, 10, 1]
rollback_similarity_threshold = 0.975
use_disjoint_test_set = True # only set for local tests

game = HangmanLocal(
    use_ngrams=use_ngrams,
    use_entropy=use_entropy,
    use_bert=use_bert,
    use_rollback=use_rollback,
    use_noncumulative_frequency=use_noncumulative_frequency,
    max_n_of_ngrams=max_n_of_ngrams,
    ngram_weights=ngram_weights,
    max_n_unknowns_for_bert=max_n_unknowns_for_bert,
    entropy_params=entropy_params,
    rollback_similarity_threshold=rollback_similarity_threshold,
)

In [4]:
# load the test set candidate secret words
if use_disjoint_test_set:
    with open("datasets/words_test_disjoint.txt", "r") as text_file:
        test_words = text_file.read().splitlines()
else: # use the training set as the test set
    with open("datasets/words_250000_train_cleaned.txt", "r") as text_file:
        test_words = text_file.read().splitlines()

print("Total Number of Words for Testing: {0}".format(len(test_words)))

# Initialize the dictionaries to hold win/loss counts per word length and unique letters
results_by_length = defaultdict(lambda: {'wins': 0, 'games': 0})
results_by_unique_letters = defaultdict(lambda: {'wins': 0, 'games': 0})

final_rates = []
# Run multiple simulations
for sim in range(1):
    print("Running Simulation {0}...".format(sim + 1))
    random.shuffle(test_words)

    test_times = 1000
    win = 0
    for i in tqdm(range(test_times)):
        word = test_words[i]
        game_result = game.start_game(word, verbose=False)
        if game_result:
            win += 1

        # Get word length and number of unique letters
        word_length = len(word)
        num_unique_letters = len(set(word))

        # Update statistics for word length
        results_by_length[word_length]['games'] += 1
        if game_result:
            results_by_length[word_length]['wins'] += 1

        # Update statistics for number of unique letters
        results_by_unique_letters[num_unique_letters]['games'] += 1
        if game_result:
            results_by_unique_letters[num_unique_letters]['wins'] += 1

    print("Success Rate: {0}/{1}={2}".format(win, test_times, win / test_times))
    final_rates.append(win / test_times)

print("Average Success Rate: {0}".format(sum(final_rates) / len(final_rates)))

# Calculate Success Rates by word length
print("\nSuccess Rate by Word Length:")
for length in sorted(results_by_length.keys()):
    data = results_by_length[length]
    win_rate = data['wins'] / data['games'] if data['games'] > 0 else 0
    print(f"Length {length}: {win_rate:.2%} ({data['wins']}/{data['games']})")

# Calculate Success Rates by number of unique letters
print("\nSuccess Rate by Number of Unique Letters:")
for num_letters in sorted(results_by_unique_letters.keys()):
    data = results_by_unique_letters[num_letters]
    win_rate = data['wins'] / data['games'] if data['games'] > 0 else 0
    print(f"Unique Letters {num_letters}: {win_rate:.2%} ({data['wins']}/{data['games']})")

Total Number of Words for Testing: 185152
Running Simulation 1...


  0%|          | 0/1000 [00:00<?, ?it/s]BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
100%|██████████| 1000/1000 [20:02<00:00,  1.20s/it]

Success Rate: 707/1000=0.707
Average Success Rate: 0.707

Success Rate by Word Length:
Length 2: 0.00% (0/3)
Length 3: 0.00% (0/3)
Length 4: 31.25% (5/16)
Length 5: 25.64% (10/39)
Length 6: 37.36% (34/91)
Length 7: 54.10% (66/122)
Length 8: 65.31% (96/147)
Length 9: 71.24% (109/153)
Length 10: 86.92% (93/107)
Length 11: 90.24% (74/82)
Length 12: 87.95% (73/83)
Length 13: 95.65% (44/46)
Length 14: 95.00% (38/40)
Length 15: 93.55% (29/31)
Length 16: 94.44% (17/18)
Length 17: 100.00% (8/8)
Length 18: 100.00% (4/4)
Length 19: 100.00% (2/2)
Length 20: 100.00% (4/4)
Length 21: 100.00% (1/1)

Success Rate by Number of Unique Letters:
Unique Letters 2: 0.00% (0/3)
Unique Letters 3: 0.00% (0/5)
Unique Letters 4: 36.36% (16/44)
Unique Letters 5: 45.65% (42/92)
Unique Letters 6: 60.99% (111/182)
Unique Letters 7: 63.55% (129/203)
Unique Letters 8: 81.48% (154/189)
Unique Letters 9: 85.71% (108/126)
Unique Letters 10: 92.31% (84/91)
Unique Letters 11: 94.44% (34/36)
Unique Letters 12: 100.00% (21/